# Bases de données — le cours à lire avant les exercices

Ce cours se lit **avant** d'attaquer les exercices du polycopié. Il part de zéro : il suffit de savoir ce qu'est un tableau de données (un fichier CSV, vu en Première).

Il suit le **programme officiel** de Terminale NSI (thème « Bases de données », publié par le ministère de l'Éducation nationale), dont voici les capacités attendues :

| Notion du programme | Ce qu'il faut savoir faire | Dans ce cours |
|:---|:---|:---|
| Modèle relationnel : relation, attribut, domaine, clef primaire, clef étrangère, schéma relationnel | identifier ces notions | parties 1, 9 et 10 |
| Base de données relationnelle | distinguer la **structure** d'une base de son **contenu** ; repérer des anomalies (redondances) dans un schéma | parties 1 et 11 |
| Système de gestion de bases de données (SGBD) | connaître ses services : persistance, accès concurrents, efficacité, sécurité | partie 2 |
| Langage SQL | construire des requêtes avec `SELECT`, `FROM`, `WHERE`, `JOIN` ; `INSERT`, `UPDATE`, `DELETE` ; on peut utiliser `DISTINCT`, `ORDER BY` et les fonctions d'agrégation, **sans** `GROUP BY` ni `HAVING` | parties 4 à 8, 10 et 12 |

**Comment lire ce cours :** chaque notion est expliquée, puis montrée sur un **exemple tapé dans la console H2** (le logiciel utilisé en classe). Le résultat affiché sous chaque exemple est celui qu'a réellement donné H2. Le mieux est de retaper les exemples soi-même.

**Plan :**
1. Le vocabulaire : relation, attribut, enregistrement, domaine
2. Le SGBD et la console H2
3. Les types (domaines de valeurs)
4. Créer une table : `CREATE TABLE`
5. Remplir une table : `INSERT INTO`, et `RUNSCRIPT`
6. Les outils Python de l'exercice 1 : lire un CSV, écrire un fichier
7. Interroger une table : `SELECT … FROM … WHERE …`
8. Compter, moyenne, minimum, maximum
9. Clé primaire
10. Plusieurs tables : clé étrangère et jointure `JOIN`
11. Pourquoi plusieurs tables ? (les anomalies)
12. Modifier et supprimer : `UPDATE`, `DELETE`
13. La méthode pour écrire une requête, et la fiche récapitulative

---
# 1. Le vocabulaire

Une **base de données relationnelle** range les données dans des **tableaux**. Exemple : un tableau des pays.

| id | code | name | continent |
|:---|:---|:---|:---|
| 1 | FR | France | EU |
| 2 | ES | Spain | EU |
| 3 | BR | Brazil | SA |

Les mots à connaître (le cours du polycopié emploie les deux colonnes) :

| Mot du cours | Mot de tous les jours | Dans l'exemple |
|:---|:---|:---|
| **relation** | table, tableau | la table `Countries` |
| **attribut** | colonne | `id`, `code`, `name`, `continent` |
| **enregistrement** (ou n-uplet) | ligne | `1, FR, France, EU` |
| **domaine** d'un attribut | les valeurs autorisées, son **type** | `id` : un entier ; `name` : du texte |

- Chaque **relation** décrit une sorte d'objets du monde réel (des pays, des aéroports, des élèves…). Chaque **ligne** est un objet ; chaque **colonne** est une caractéristique de ces objets.
- **Structure et contenu** : la **structure** d'une table, c'est la liste de ses colonnes et de leurs types (elle change rarement) ; le **contenu**, ce sont les lignes (elles changent tout le temps). On décrit la structure par un **schéma** :

  Countries(id : INT, code : CHAR(2), name : VARCHAR, continent : VARCHAR)

- **Pourquoi pas un simple fichier CSV ?** Un CSV convient pour quelques centaines de lignes lues par un seul programme. Une base de données gère des millions de lignes, utilisées **en même temps** par beaucoup d'utilisateurs, en **vérifiant** que les données restent cohérentes. C'est le rôle du SGBD.

---
# 2. Le SGBD et la console H2

Un **SGBD** (système de gestion de bases de données) est le logiciel qui stocke la base et répond aux demandes. En classe, on utilise le SGBD **H2**. D'autres exemples : SQLite, MySQL, PostgreSQL, Oracle.

**Les services rendus par un SGBD** (à savoir citer) :
- **persistance** : les données sont enregistrées sur le disque, elles restent quand on éteint l'ordinateur ;
- **gestion des accès concurrents** : plusieurs utilisateurs peuvent lire et modifier la base **en même temps** sans tout mélanger (deux personnes ne peuvent pas réserver le même siège) ;
- **efficacité** : il trouve très vite une ligne parmi des millions ;
- **sécurisation des accès** : chaque utilisateur a un mot de passe et des droits (lire seulement, ou modifier) ;
- en plus, il **vérifie la cohérence** des données (les *contraintes d'intégrité*, parties 3, 9 et 10).

**On parle au SGBD avec le langage SQL.** Chaque demande s'appelle une **requête** (ou instruction) et se termine par un **point-virgule** `;`. Les majuscules aux mots-clés (`SELECT`, `FROM`…) sont une habitude de lecture, pas une obligation. Les retours à la ligne et les espaces n'ont pas d'importance.

**Utiliser la console H2 :**
1. lancer « H2 Console » : une page s'ouvre dans le navigateur ;
2. se connecter : JDBC URL `jdbc:h2:~/nsi` (le fichier de la base, dans le dossier personnel), utilisateur `sa`, mot de passe vide, bouton **Connecter** ;
3. taper la requête dans la grande zone de texte, puis cliquer sur **Exécuter** (ou Ctrl + Entrée) ;
4. le résultat s'affiche en dessous, sous forme de tableau.

On peut taper plusieurs requêtes d'un coup (chacune finie par `;`) : H2 les exécute dans l'ordre.

> Pour repartir d'une base vide : `DROP ALL OBJECTS;` efface toutes les tables de la base connectée.

---
# 3. Les types (domaines de valeurs)

Chaque colonne a un **type** : le SGBD **refuse** une valeur qui n'est pas du bon type. C'est la **contrainte de domaine**.

| Type | Contenu | Exemple de valeur en SQL |
|:---|:---|:---|
| `INT` | nombre entier | `302687` |
| `REAL` (ou `FLOAT`) | nombre à virgule (avec un **point**) | `48.72` |
| `CHAR(n)` | texte d'**au plus** n caractères | `'FR'` pour `CHAR(2)` |
| `VARCHAR` | texte de longueur quelconque | `'France'` |
| `DATE` | une date | `'2026-11-14'` (année-mois-jour) |
| `TIMESTAMP` | une date et une heure | `'2026-11-14 08:30:00'` |
| `BOOLEAN` | vrai ou faux | `TRUE`, `FALSE` |

⚠️ **Le texte s'écrit entre apostrophes** `'...'` (pas entre guillemets). Tout ce qui est entre les apostrophes compte, **espaces compris** : `' FR '` fait 4 caractères. Les nombres s'écrivent **sans** apostrophes.

---
# 4. Créer une table : `CREATE TABLE`

On décrit la **structure** : le nom de la table, puis chaque colonne avec son type, séparées par des virgules (pas de virgule après la dernière).

*Dans la console H2 :*

```sql
CREATE TABLE Countries (
    id INT,
    code CHAR(2),
    name VARCHAR,
    continent VARCHAR );
```

*Résultat dans la console H2 :*

```text
(Update count: 0)
```

- `Update count: 0` : la table est créée, elle est **vide** (aucune ligne modifiée).
- Si on l'exécute une deuxième fois, H2 répond que la table existe déjà. Pour la supprimer (structure **et** contenu) : `DROP TABLE Countries;`.
- C'est exactement la question 1 de l'exercice 1 du polycopié.

---
# 5. Remplir une table : `INSERT INTO`

Une requête `INSERT INTO` ajoute **une ligne**. On donne les colonnes, puis les valeurs **dans le même ordre**.

*Dans la console H2 :*

```sql
INSERT INTO Countries (id, code, name, continent) VALUES (1, 'FR', 'France', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (2, 'ES', 'Spain', 'EU');
INSERT INTO Countries (id, code, name, continent) VALUES (3, 'BR', 'Brazil', 'SA');
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Countries (id, code, name, continent) VALUES (1, 'FR', 'France', 'EU');
(Update count: 1)

INSERT INTO Countries (id, code, name, continent) VALUES (2, 'ES', 'Spain', 'EU');
(Update count: 1)

INSERT INTO Countries (id, code, name, continent) VALUES (3, 'BR', 'Brazil', 'SA');
(Update count: 1)

SELECT * FROM Countries;
ID | CODE | NAME   | CONTINENT
1  | FR   | France | EU
2  | ES   | Spain  | EU
3  | BR   | Brazil | SA
(3 rows)
```

- `Update count: 1` : une ligne ajoutée. `SELECT * FROM Countries;` affiche toute la table (partie 7).
- Si on donne une valeur pour **toutes** les colonnes, dans l'ordre de la table, on peut omettre la liste des colonnes : `INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');`.

**Le SGBD vérifie les types.** Une chaîne de 4 caractères dans une colonne `CHAR(2)` est refusée :

*Dans la console H2 :*

```sql
INSERT INTO Countries VALUES (4, ' IM ', 'Isle of Man', 'EU');
```

*Résultat dans la console H2 :*

```text
Erreur : Valeur trop longue pour la colonne "CODE CHARACTER(2)": "' IM ' (4)"
Value too long for column "CODE CHARACTER(2)": "' IM ' (4)"; SQL statement:
INSERT INTO Countries VALUES (4, ' IM ', 'Isle of Man', 'EU') [22001-252]
```

⚠️ C'est **le piège du copier-coller depuis le PDF** du polycopié : il ajoute des espaces autour des apostrophes. Il faut taper `'IM'`, collé aux apostrophes.

**Une apostrophe dans un texte** (*Côte d'Ivoire*) : comme l'apostrophe termine le texte, on la **double** : `'Côte d''Ivoire'`.

*Dans la console H2 :*

```sql
INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');
INSERT INTO Countries VALUES (5, 'CI', 'Côte d''Ivoire', 'AF');
INSERT INTO Countries VALUES (6, 'JP', 'Japan', 'AS');
SELECT * FROM Countries;
```

*Résultat dans la console H2 :*

```text
INSERT INTO Countries VALUES (4, 'IM', 'Isle of Man', 'EU');
(Update count: 1)

INSERT INTO Countries VALUES (5, 'CI', 'Côte d''Ivoire', 'AF');
(Update count: 1)

INSERT INTO Countries VALUES (6, 'JP', 'Japan', 'AS');
(Update count: 1)

SELECT * FROM Countries;
ID | CODE | NAME          | CONTINENT
1  | FR   | France        | EU
2  | ES   | Spain         | EU
3  | BR   | Brazil        | SA
4  | IM   | Isle of Man   | EU
5  | CI   | Côte d'Ivoire | AF
6  | JP   | Japan         | AS
(6 rows)
```

### Exécuter un fichier : `RUNSCRIPT`

Un fichier `.sql` est un simple **fichier texte** qui contient des requêtes, l'une après l'autre. Au lieu de les recopier, on demande à H2 de toutes les exécuter :

```sql
RUNSCRIPT FROM 'C:/Users/Iheb/Downloads/countries.sql';
```

(entre apostrophes, le chemin complet du fichier, avec des `/` même sous Windows ; sur Mac : `'/Users/iheb/Downloads/countries.sql'`). C'est ainsi qu'on charge les données des exercices 2 (aéroports) et 3 (Vélib').

---
# 6. Les outils Python de l'exercice 1

La question 2 (b) de l'exercice 1 demande un programme Python qui **fabrique** le fichier `countries.sql` à partir du fichier `countries.csv`. Il faut trois outils.

### Outil 1 : écrire dans un fichier texte

```python
fichier = open("essai.txt", "w", encoding="utf-8")   # "w" = write : on crée le fichier (ou on l'efface)
fichier.write("première ligne\n")                      # "\n" = passer à la ligne
fichier.write("deuxième ligne\n")
fichier.close()                                         # obligatoire : c'est là que tout est enregistré
```
`write` écrit **exactement** la chaîne donnée : sans `"\n"`, tout serait sur la même ligne.

### Outil 2 : lire un fichier CSV avec le module `csv`

**Étape 1 — Ce qu'est un fichier CSV.** C'est un simple **fichier texte** qui contient un tableau. Chaque ligne du fichier est une ligne du tableau, et les valeurs sont séparées par des **virgules**. La première ligne donne les **noms des colonnes**.

Exemple : un petit fichier `eleves.csv` de 3 lignes.

```text
nom,prenom,classe
Martin,Léa,T3
Durand,Paul,T1
```
Il représente ce tableau :

| nom | prenom | classe |
|:---|:---|:---|
| Martin | Léa | T3 |
| Durand | Paul | T1 |

**Étape 2 — Le problème.** Si on lit ce fichier normalement, Python donne chaque ligne comme **un seul bloc de texte** :
```python
fichier = open("eleves.csv", encoding="utf-8")
for ligne in fichier:
    print(repr(ligne))
fichier.close()
```
```text
'nom,prenom,classe
'
'Martin,Léa,T3
'
'Durand,Paul,T1
'
```
Pour avoir le prénom de Léa, il faudrait découper le texte aux virgules soi-même. C'est pénible : c'est le module `csv` qui fait ce travail.

**Étape 3 — La solution : `csv.DictReader`.** Il lit le fichier **ligne par ligne** et, pour chaque ligne du tableau, il donne un **dictionnaire** :
- les **clés** sont les noms des colonnes (pris dans la première ligne du fichier) ;
- les **valeurs** sont le contenu de la ligne.

```python
import csv

fichier = open("eleves.csv", encoding="utf-8")
for ligne in csv.DictReader(fichier):
    print(ligne)
fichier.close()
```
```text
{'nom': 'Martin', 'prenom': 'Léa', 'classe': 'T3'}
{'nom': 'Durand', 'prenom': 'Paul', 'classe': 'T1'}
```
La boucle fait **2 tours** (un par élève). La ligne d'en-tête ne donne pas de tour : elle sert seulement à fabriquer les clés.

> **Image à retenir :** `DictReader` est comme quelqu'un qui lit le tableau à ta place et te tend, à chaque tour, une **fiche** : « nom → Martin, prénom → Léa, classe → T3 ».

**Étape 4 — Utiliser la fiche.** Comme `ligne` est un dictionnaire, on récupère une valeur avec le **nom de la colonne** entre crochets :
```python
fichier = open("eleves.csv", encoding="utf-8")
for ligne in csv.DictReader(fichier):
    print(ligne["prenom"], "est en", ligne["classe"])
fichier.close()
```
```text
Léa est en T3
Paul est en T1
```

| tour de boucle | `ligne` | `ligne["prenom"]` | `ligne["classe"]` |
|:---|:---|:---|:---|
| 1 | `{'nom': 'Martin', 'prenom': 'Léa', 'classe': 'T3'}` | `'Léa'` | `'T3'` |
| 2 | `{'nom': 'Durand', 'prenom': 'Paul', 'classe': 'T1'}` | `'Paul'` | `'T1'` |

**Étape 5 — Avec le vrai fichier de l'exercice 1.** C'est exactement pareil avec `countries.csv`, qui a les colonnes `id`, `code`, `name`, `continent` :
```text
"id","code","name","continent"
302672,"AD","Andorra","EU"
```
Au premier tour, `DictReader` donne :
```text
{'id': '302672', 'code': 'AD', 'name': 'Andorra', 'continent': 'EU'}
```
- les guillemets du fichier ont disparu : `csv` les enlève tout seul ;
- ⚠️ **toutes les valeurs sont des chaînes**, même l'identifiant : `'302672'` et pas `302672`. Ici, c'est pratique : on peut directement les coller avec `+` (outil 3).

### Outil 3 : fabriquer une requête SQL avec `+`

On colle des morceaux de texte avec `+` :
```python
ligne = "INSERT INTO Countries VALUES (" + pays["id"] + ", '" + pays["code"] + "', '" + pays["name"] + "', '" + pays["continent"] + "');"
```
Pour Andorre, `ligne` vaut : `INSERT INTO Countries VALUES (302672, 'AD', 'Andorra', 'EU');`

Regarder les guillemets de près : la chaîne Python `", '"` contient une virgule, une espace et **une apostrophe** — c'est cette apostrophe qui ouvre le texte SQL `'AD'`.

⚠️ **Le piège de l'apostrophe** : pour *Côte d'Ivoire*, on obtiendrait `'Côte d'Ivoire'`, refusé par H2. Il faut doubler l'apostrophe (partie 5) avant de construire la ligne, avec une petite fonction qui parcourt le texte caractère par caractère :
```python
def echappe(texte):
    resultat = ""
    for caractere in texte:
        if caractere == "'":
            resultat = resultat + "''"
        else:
            resultat = resultat + caractere
    return resultat
```

**Avec ces trois outils**, le programme de l'exercice 1 : ouvrir le CSV en lecture et `countries.sql` en écriture ; pour chaque pays, construire la ligne `INSERT` (avec `echappe` sur le nom) et l'écrire suivie de `"\n"` ; fermer les deux fichiers.

---
# 7. Interroger une table : `SELECT … FROM … WHERE …`

Jusqu'ici, on a **rangé** des données. Maintenant, on va **poser des questions** à la base : « quels aéroports sont à Paris ? », « combien y a-t-il d'aéroports en France ? ». Une question posée en SQL s'appelle une **requête d'interrogation**. Elle commence toujours par `SELECT`.

**L'idée la plus importante de cette partie : une requête `SELECT` ne modifie rien.** Elle **lit** une table et **fabrique un nouveau tableau**, le résultat, que H2 affiche. La table elle-même reste intacte.

### La table qui va servir d'exemple

Pour s'entraîner, on crée une table `Airports` de 7 aéroports. Avant chaque requête, on pourra regarder ce tableau et **deviner le résultat**, puis vérifier avec H2.

*On crée et on remplit la table `Airports` (rien de nouveau : parties 4 et 5) :*

```sql
CREATE TABLE Airports (
    code CHAR(4), name VARCHAR, type VARCHAR,
    latitude REAL, altitude INT, country CHAR(2), city VARCHAR );
INSERT INTO Airports VALUES ('LFPG', 'Charles de Gaulle Airport', 'large_airport', 49.01, 392, 'FR', 'Paris');
INSERT INTO Airports VALUES ('LFPO', 'Paris-Orly Airport', 'large_airport', 48.72, 291, 'FR', 'Paris');
INSERT INTO Airports VALUES ('LFBO', 'Toulouse-Blagnac Airport', 'large_airport', 43.63, 499, 'FR', 'Toulouse');
INSERT INTO Airports VALUES ('LFPI', 'Paris Issy Heliport', 'heliport', 48.83, 112, 'FR', 'Paris');
INSERT INTO Airports VALUES ('LEMD', 'Madrid-Barajas Airport', 'large_airport', 40.47, 1998, 'ES', 'Madrid');
INSERT INTO Airports VALUES ('SBGR', 'Guarulhos Airport', 'large_airport', -23.44, 2459, 'BR', 'Sao Paulo');
INSERT INTO Airports VALUES ('RJTT', 'Tokyo Haneda Airport', 'large_airport', 35.55, 35, 'JP', 'Tokyo');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Airports ( code CHAR(4), name VARCHAR, type VARCHAR, latitude REAL, altitude INT, country CHAR(2), city VARCHAR );
(Update count: 0)

INSERT INTO Airports VALUES ('LFPG', 'Charles de Gaulle Airport', 'large_airport', 49.01, 392, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES ('LFPO', 'Paris-Orly Airport', 'large_airport', 48.72, 291, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES ('LFBO', 'Toulouse-Blagnac Airport', 'large_airport', 43.63, 499, 'FR', 'Toulouse');
(Update count: 1)

INSERT INTO Airports VALUES ('LFPI', 'Paris Issy Heliport', 'heliport', 48.83, 112, 'FR', 'Paris');
(Update count: 1)

INSERT INTO Airports VALUES ('LEMD', 'Madrid-Barajas Airport', 'large_airport', 40.47, 1998, 'ES', 'Madrid');
(Update count: 1)

INSERT INTO Airports VALUES ('SBGR', 'Guarulhos Airport', 'large_airport', -23.44, 2459, 'BR', 'Sao Paulo');
(Update count: 1)

INSERT INTO Airports VALUES ('RJTT', 'Tokyo Haneda Airport', 'large_airport', 35.55, 35, 'JP', 'Tokyo');
(Update count: 1)
```

Voici donc la table `Airports` (à garder sous les yeux pour toute la suite) :

| code | name | type | latitude | altitude | country | city |
|:---|:---|:---|:---|:---|:---|:---|
| LFPG | Charles de Gaulle Airport | large_airport | 49.01 | 392 | FR | Paris |
| LFPO | Paris-Orly Airport | large_airport | 48.72 | 291 | FR | Paris |
| LFBO | Toulouse-Blagnac Airport | large_airport | 43.63 | 499 | FR | Toulouse |
| LFPI | Paris Issy Heliport | heliport | 48.83 | 112 | FR | Paris |
| LEMD | Madrid-Barajas Airport | large_airport | 40.47 | 1998 | ES | Madrid |
| SBGR | Guarulhos Airport | large_airport | -23.44 | 2459 | BR | Sao Paulo |
| RJTT | Tokyo Haneda Airport | large_airport | 35.55 | 35 | JP | Tokyo |

(`latitude` : positive au nord de l'équateur, négative au sud ; `altitude` en pieds ; `country` : le code du pays.)

---
## 7.1 Choisir les colonnes : `SELECT … FROM …`

**Le besoin :** « Je veux la liste des noms d'aéroports et de leurs villes. » On ne veut pas tout le tableau, seulement **deux colonnes**.

**La syntaxe :**
```sql
SELECT name, city FROM Airports;
```
Elle se lit comme une phrase :

| morceau | sens |
|:---|:---|
| `SELECT name, city` | « montre-moi les colonnes `name` et `city` » (séparées par une virgule) |
| `FROM Airports` | « en les prenant dans la table `Airports` » |
| `;` | fin de la requête |

**Ce que fait H2 :** il prend la table, **garde seulement les colonnes demandées**, et garde **toutes les lignes**. Devine le résultat, puis vérifie :

*Dans la console H2 :*

```sql
SELECT name, city FROM Airports;
```

*Résultat dans la console H2 :*

```text
NAME                      | CITY
Charles de Gaulle Airport | Paris
Paris-Orly Airport        | Paris
Toulouse-Blagnac Airport  | Toulouse
Paris Issy Heliport       | Paris
Madrid-Barajas Airport    | Madrid
Guarulhos Airport         | Sao Paulo
Tokyo Haneda Airport      | Tokyo
(7 rows)
```

- Il y a bien 7 lignes, mais 2 colonnes seulement. H2 écrit les noms de colonnes en majuscules.
- `SELECT * FROM Airports;` : l'étoile `*` veut dire « **toutes** les colonnes ». C'est la requête à taper en premier pour **découvrir** une table.

✏️ **Vérifie que tu as compris :** quelle requête affiche seulement les codes et les altitudes ? *(Réponse : `SELECT code, altitude FROM Airports;`)*

---
## 7.2 Choisir les lignes : `WHERE`

**Le besoin :** « Je veux les noms des aéroports **de Paris**. » Cette fois, on ne veut pas toutes les lignes : seulement celles qui vérifient une **condition**.

**L'idée :** `WHERE` est un **filtre**. H2 regarde les lignes **une par une**, teste la condition sur chacune, et ne garde que celles où elle est **vraie**. Pour la condition `city = 'Paris'` :

| code | city | `city = 'Paris'` ? | gardée ? |
|:---|:---|:---|:---|
| LFPG | Paris | vrai | ✔ |
| LFPO | Paris | vrai | ✔ |
| LFBO | Toulouse | faux | ✘ |
| LFPI | Paris | vrai | ✔ |
| LEMD | Madrid | faux | ✘ |
| SBGR | Sao Paulo | faux | ✘ |
| RJTT | Tokyo | faux | ✘ |

Puis `SELECT` choisit les colonnes à afficher, **parmi les lignes gardées**.

**La syntaxe :** on ajoute `WHERE condition` **après** le `FROM`.
```sql
SELECT name FROM Airports WHERE city = 'Paris';
```

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE city = 'Paris';
```

*Résultat dans la console H2 :*

```text
NAME
Charles de Gaulle Airport
Paris-Orly Airport
Paris Issy Heliport
(3 rows)
```

**Les comparaisons possibles :**

| en SQL | sens | exemple |
|:---|:---|:---|
| `=` | égal (un seul `=`, pas `==` comme en Python) | `country = 'FR'` |
| `<>` | différent | `type <> 'heliport'` |
| `<`, `>`, `<=`, `>=` | plus petit, plus grand… | `altitude > 1000` |

Les textes sont entre apostrophes (`'Paris'`), les nombres sans (`1000`).

**Plusieurs conditions :** on les relie avec `AND` (les deux doivent être vraies), `OR` (au moins une), comme `and` et `or` en Python.

*« Les grands aéroports de Paris »* : il faut être à Paris **et** être un grand aéroport. L'héliport de Paris est écarté.

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE city = 'Paris' AND type = 'large_airport';
```

*Résultat dans la console H2 :*

```text
NAME
Charles de Gaulle Airport
Paris-Orly Airport
(2 rows)
```

*« Les aéroports très hauts (plus de 1000 pieds) ou au Japon »* : il suffit d'une des deux conditions.

*Dans la console H2 :*

```sql
SELECT code, altitude FROM Airports WHERE altitude > 1000 OR country = 'JP';
```

*Résultat dans la console H2 :*

```text
CODE | ALTITUDE
LEMD | 1998
SBGR | 2459
RJTT | 35
(3 rows)
```

**L'ordre dans lequel H2 travaille** (différent de l'ordre d'écriture !) :
1. `FROM` : quelle table ?
2. `WHERE` : quelles lignes garder ?
3. `SELECT` : quelles colonnes afficher ?

C'est aussi l'ordre dans lequel il faut **lire** une requête pour la comprendre.

✏️ **Vérifie que tu as compris :** sans H2, quel est le résultat de `SELECT city FROM Airports WHERE country = 'FR' AND altitude < 300;` ? *(Réponse : deux lignes, Paris (Orly, 291) et Paris (Issy, 112).)*

---
## 7.3 Chercher un mot dans un texte : `LIKE`

**Le besoin :** « Les aéroports dont le nom **contient** le mot Paris. » Avec `=`, il faudrait connaître le nom **exact**. `LIKE` permet de chercher un **morceau** de texte.

**L'idée :** on écrit un **motif**, où le symbole `%` veut dire « n'importe quoi » (n'importe quels caractères, ou rien du tout).

| motif | sens | `'Paris-Orly Airport'` correspond ? |
|:---|:---|:---|
| `'Paris%'` | commence par Paris | ✔ |
| `'%Airport'` | finit par Airport | ✔ |
| `'%Orly%'` | contient Orly | ✔ |
| `'%Tokyo%'` | contient Tokyo | ✘ |

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE name LIKE '%Paris%';
```

*Résultat dans la console H2 :*

```text
NAME
Paris-Orly Airport
Paris Issy Heliport
(2 rows)
```

- « Charles de Gaulle Airport » n'est **pas** dans le résultat : son nom ne contient pas le mot Paris (même si l'aéroport est à Paris !). On cherche dans le **texte** de la colonne `name`, pas dans le sens.
- ⚠️ Dans H2, `LIKE` **distingue majuscules et minuscules** : `'%paris%'` ne trouverait rien ici.

---
## 7.4 Trier les résultats : `ORDER BY`

**Le besoin :** « Je veux les aéroports français, triés selon leur altitude, de la plus haute à la plus basse. » On sait déjà garder les aéroports français (avec `WHERE`) et afficher leur nom et leur altitude (avec `SELECT`). Ce qui est nouveau, c'est l'**ordre** des lignes.

**Étape 1 — Ce que donne la requête sans tri.** H2 affiche les lignes dans l'ordre où elles ont été enregistrées dans la table :

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
NAME                      | ALTITUDE
Charles de Gaulle Airport | 392
Paris-Orly Airport        | 291
Toulouse-Blagnac Airport  | 499
Paris Issy Heliport       | 112
(4 rows)
```

Les altitudes arrivent dans le désordre : 392, 291, 499, 112. On ne voit pas tout de suite quel aéroport est le plus haut.

**Étape 2 — Trier.** **Trier**, c'est ranger les lignes du résultat selon les valeurs d'une colonne. En SQL, on ajoute **à la fin** de la requête `ORDER BY` (« trier selon ») suivi du nom de la colonne qui sert au tri. Ici, on trie selon l'altitude :

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR' ORDER BY altitude;
```

*Résultat dans la console H2 :*

```text
NAME                      | ALTITUDE
Paris Issy Heliport       | 112
Paris-Orly Airport        | 291
Charles de Gaulle Airport | 392
Toulouse-Blagnac Airport  | 499
(4 rows)
```

Les lignes sont maintenant triées dans l'**ordre croissant** : du plus petit au plus grand (112, 291, 392, 499). C'est l'ordre choisi par défaut, quand on n'écrit rien après le nom de la colonne. Pour une colonne de texte, l'ordre croissant est l'ordre alphabétique, de A à Z. Plus précisément, H2 compare les caractères un par un, et les majuscules passent avant les minuscules : `Zebra` vient avant `avion`.

**Étape 3 — Choisir le sens du tri.** On voulait l'inverse : du plus haut au plus bas, c'est-à-dire l'**ordre décroissant** (du plus grand au plus petit). On ajoute le mot `DESC` (de l'anglais *descending*, « décroissant ») après le nom de la colonne :

*Dans la console H2 :*

```sql
SELECT name, altitude FROM Airports WHERE country = 'FR' ORDER BY altitude DESC;
```

*Résultat dans la console H2 :*

```text
NAME                      | ALTITUDE
Toulouse-Blagnac Airport  | 499
Charles de Gaulle Airport | 392
Paris-Orly Airport        | 291
Paris Issy Heliport       | 112
(4 rows)
```

On obtient bien les aéroports français du plus haut (Toulouse, 499) au plus bas (l'héliport d'Issy, 112).

**À retenir :**

| on écrit à la fin de la requête | tri obtenu |
|:---|:---|
| `ORDER BY colonne` | croissant : du plus petit au plus grand, de A à Z |
| `ORDER BY colonne DESC` | décroissant : du plus grand au plus petit, de Z à A |

`ORDER BY` ne change pas **quelles** lignes sont affichées (c'est le rôle de `WHERE`) : il change seulement **l'ordre** dans lequel elles sont affichées.

---
## 7.5 Enlever les doublons : `DISTINCT`

**Le besoin :** « Dans quelles villes françaises y a-t-il un aéroport ? »

**Étape 1 — La requête naturelle.** On veut la colonne `city`, pour les aéroports français :

*Dans la console H2 :*

```sql
SELECT city FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
CITY
Paris
Paris
Toulouse
Paris
(4 rows)
```

**Étape 2 — Le problème.** Paris apparaît **trois fois**. Ce n'est pas une erreur : il y a trois aéroports français à Paris (Charles de Gaulle, Orly et l'héliport d'Issy), et H2 affiche une ligne par aéroport gardé. Une valeur qui apparaît plusieurs fois dans un résultat s'appelle un **doublon**. Or la question demande les **villes**, et chaque ville ne devrait apparaître qu'une fois.

**Étape 3 — `DISTINCT`.** On écrit le mot `DISTINCT` juste après `SELECT`. H2 affiche alors chaque valeur **une seule fois** (*distinct* veut dire « différent » : on ne garde que des valeurs différentes) :

*Dans la console H2 :*

```sql
SELECT DISTINCT city FROM Airports WHERE country = 'FR';
```

*Résultat dans la console H2 :*

```text
CITY
Paris
Toulouse
(2 rows)
```

Il reste deux lignes : Paris et Toulouse. C'est la réponse à la question.

**À retenir :** quand une question demande « les différents… », « quelles villes… », « quels types… », on se demande si une même valeur peut revenir plusieurs fois dans le résultat. Si oui, on écrit `SELECT DISTINCT`.

**L'ordre d'écriture complet** d'une requête d'interrogation (on peut sauter les parties inutiles, mais pas changer l'ordre) :
```sql
SELECT [DISTINCT] colonnes
FROM table
WHERE condition
ORDER BY colonne [DESC];
```

---
# 8. Résumer une colonne en un seul nombre : les fonctions d'agrégation

**Le besoin :** parfois on ne veut pas une **liste**, mais **un seul nombre** qui résume : « **combien** y a-t-il d'aéroports en France ? », « quelle est l'altitude **moyenne** ? », « quelle est la **plus grande** altitude ? ».

**L'idée :** une fonction d'agrégation prend **toute une colonne** (toutes les lignes gardées par le `WHERE`) et la **résume en une seule valeur**.

| fonction | ce qu'elle calcule | sur la colonne altitude des 7 aéroports |
|:---|:---|:---|
| `COUNT(*)` | le **nombre** de lignes | 7 |
| `SUM(altitude)` | la **somme** | 5786 |
| `AVG(altitude)` | la **moyenne** (average) | 5786 ÷ 7 ≈ 826,6 |
| `MIN(altitude)` | la **plus petite** valeur | 35 |
| `MAX(altitude)` | la **plus grande** valeur | 2459 |

**Avec un `WHERE`**, le filtre agit **d'abord**, puis la fonction résume les lignes qui restent. *« Combien d'aéroports en France ? »* : on garde les 4 lignes françaises, puis `COUNT(*)` les compte.

*Dans la console H2 :*

```sql
SELECT COUNT(*) FROM Airports WHERE country = 'FR';
SELECT AVG(altitude) FROM Airports;
SELECT MIN(latitude), MAX(altitude) FROM Airports;
```

*Résultat dans la console H2 :*

```text
SELECT COUNT(*) FROM Airports WHERE country = 'FR';
COUNT(*)
4
(1 row)

SELECT AVG(altitude) FROM Airports;
AVG(ALTITUDE)
826.5714285714286
(1 row)

SELECT MIN(latitude), MAX(altitude) FROM Airports;
MIN(LATITUDE) | MAX(ALTITUDE)
-23.44        | 2459
(1 row)
```

### Obtenir une ligne entière, et pas seulement un nombre

**Le besoin :** « Quel est le **nom** de l'aéroport le plus au sud ? »

Sur Terre, plus on va vers le sud, plus la latitude est petite : elle devient négative au sud de l'équateur. L'aéroport le plus au sud est donc celui qui a la **plus petite latitude**. On vient de voir que `MIN(latitude)` donne cette plus petite latitude, -23.44. Mais la question ne demande pas ce nombre : elle demande le **nom** de l'aéroport qui a cette latitude.

**Étape 1 — L'idée qui vient naturellement, et pourquoi elle ne marche pas.** On a envie de demander le nom et la latitude minimale dans la même requête :

*Dans la console H2 :*

```sql
SELECT name, MIN(latitude) FROM Airports;
```

*Résultat dans la console H2 :*

```text
Erreur : La colonne "NAME" doit être dans la liste du GROUP BY
Column "NAME" must be in the GROUP BY list; SQL statement:
SELECT name, MIN(latitude) FROM Airports [90016-252]
```

H2 refuse. `MIN(latitude)` résume les 7 lignes en **une seule** valeur, alors que la colonne `name` contient **7** valeurs, une par aéroport. H2 ne peut pas afficher 7 noms à côté d'un seul nombre, et il ne devine pas que l'on veut « le nom qui va avec le minimum ». (Le message parle de `GROUP BY`, une clause que le programme exclut : il suffit de retenir que cette requête est refusée.)

Il existe deux méthodes pour obtenir la réponse.

**Étape 2 — Méthode 1 : trier, puis lire la première ligne.** On trie les aéroports par latitude croissante (partie 7.4) : le plus au sud arrive en première ligne.

*Dans la console H2 :*

```sql
SELECT name, latitude FROM Airports ORDER BY latitude;
```

*Résultat dans la console H2 :*

```text
NAME                      | LATITUDE
Guarulhos Airport         | -23.44
Tokyo Haneda Airport      | 35.55
Madrid-Barajas Airport    | 40.47
Toulouse-Blagnac Airport  | 43.63
Paris-Orly Airport        | 48.72
Paris Issy Heliport       | 48.83
Charles de Gaulle Airport | 49.01
(7 rows)
```

La réponse est dans la première ligne : Guarulhos Airport. L'inconvénient : la requête affiche les 7 aéroports, et c'est à nous de lire la bonne ligne.

**Étape 3 — Méthode 2 : une requête dans une requête (sous-requête).** On fait en SQL ce qu'on ferait à la main, en deux temps :
1. on cherche d'abord la plus petite latitude : `SELECT MIN(latitude) FROM Airports` donne -23.44 ;
2. on cherche ensuite l'aéroport qui a cette latitude : `SELECT name FROM Airports WHERE latitude = -23.44`.

Plutôt que de recopier le nombre -23.44 à la main, on écrit la première requête **à sa place**, entre parenthèses :

*Dans la console H2 :*

```sql
SELECT name FROM Airports WHERE latitude = (SELECT MIN(latitude) FROM Airports);
```

*Résultat dans la console H2 :*

```text
NAME
Guarulhos Airport
(1 row)
```

H2 exécute d'abord la requête entre parenthèses. Comme toute requête `SELECT`, elle donne un tableau, mais ici un tableau d'**une seule ligne et une seule colonne**, qui contient -23.44. Placé après le `=`, ce tableau est utilisé comme la **valeur unique** qu'il contient : la requête principale revient à `WHERE latitude = -23.44`. Le résultat est **directement** la réponse, en une seule requête.

**Attention :** après un `=`, la sous-requête doit donner **une seule valeur**. C'est le cas avec `MIN`, `MAX`, `COUNT`, `SUM` et `AVG`, qui résument toute une colonne en un nombre. Si la sous-requête renvoyait plusieurs lignes, H2 refuserait la requête, avec le message « La sous-requête scalaire contient plus d'une rangée » (*scalaire* : une seule valeur ; *rangée* : ligne).

**À retenir :** une fonction d'agrégation donne un **nombre**. Pour obtenir **la ligne** qui correspond à ce nombre (le nom, la ville…), on trie avec `ORDER BY`, ou on utilise une sous-requête : `WHERE colonne = (SELECT MIN(colonne) FROM table)`.

> Le programme précise qu'on n'utilise **pas** `GROUP BY` (qui ferait un calcul « pour chaque pays » ou « pour chaque continent »). Si on a besoin d'un résultat par pays, on fait une requête par pays.

✏️ **Vérifie que tu as compris :** quelle requête donne le nombre d'héliports ? *(Réponse : `SELECT COUNT(*) FROM Airports WHERE type = 'heliport';` → 1.)*

---
# 9. La clé primaire : reconnaître chaque ligne

**Le besoin :** dans une table, chaque ligne représente **un objet du monde réel** : un pays, un aéroport, un élève… Pour que la base soit juste, il faut que chaque objet n'apparaisse **qu'une seule fois**, et qu'on puisse **désigner une ligne précise sans ambiguïté**, pour la lire ou la modifier.

**Étape 1 — Ce qui se passe si un objet est enregistré deux fois.** Rien n'empêche, dans une table comme `Countries` (partie 5), d'insérer deux fois le même pays. Pour le voir sans abîmer `Countries`, on fait l'essai sur une petite table d'essai, `PaysEssai`, créée de la même façon (sans rien de plus) :

*Dans la console H2 :*

```sql
CREATE TABLE PaysEssai (code CHAR(2), name VARCHAR, continent VARCHAR);
INSERT INTO PaysEssai VALUES ('FR', 'France', 'EU');
INSERT INTO PaysEssai VALUES ('ES', 'Spain', 'EU');
INSERT INTO PaysEssai VALUES ('FR', 'France', 'EU');
SELECT * FROM PaysEssai;
SELECT COUNT(*) FROM PaysEssai WHERE continent = 'EU';
```

*Résultat dans la console H2 :*

```text
CREATE TABLE PaysEssai (code CHAR(2), name VARCHAR, continent VARCHAR);
(Update count: 0)

INSERT INTO PaysEssai VALUES ('FR', 'France', 'EU');
(Update count: 1)

INSERT INTO PaysEssai VALUES ('ES', 'Spain', 'EU');
(Update count: 1)

INSERT INTO PaysEssai VALUES ('FR', 'France', 'EU');
(Update count: 1)

SELECT * FROM PaysEssai;
CODE | NAME   | CONTINENT
FR   | France | EU
ES   | Spain  | EU
FR   | France | EU
(3 rows)

SELECT COUNT(*) FROM PaysEssai WHERE continent = 'EU';
COUNT(*)
3
(1 row)
```

H2 a tout accepté : la France est enregistrée **deux fois**. Cela fausse la base de trois façons :
- **les calculs deviennent faux** : il n'y a que deux pays d'Europe (la France et l'Espagne), mais `COUNT(*)` en compte **3** ;
- **les informations peuvent se contredire** : si on corrige une information de la France sur une seule des deux lignes, la base dit deux choses différentes du même pays ;
- **on ne peut plus désigner une ligne précise** : les deux lignes « France » sont identiques ; aucune condition `WHERE` ne permet de choisir l'une sans l'autre.

**Étape 2 — L'idée : un identifiant.** Pour éviter cela, on choisit dans chaque table un attribut (ou un groupe d'attributs) dont la valeur est **différente pour chaque ligne**, comme un numéro de carte d'identité pour une personne.

**Définition (polycopié).** Un **identifiant** (ou **clé**) d'une relation est un attribut, ou un groupe d'attributs, qui permet d'identifier **de manière unique** un enregistrement : deux lignes ne peuvent pas avoir le même identifiant.

**Étape 3 — Choisir la clé.** Pour savoir si un attribut peut servir d'identifiant, on se pose **une seule question** : *deux objets différents peuvent-ils avoir la même valeur ?* Exemple avec la table `Airports` de la partie 7 :

| attribut des aéroports | deux aéroports peuvent-ils avoir la même valeur ? | identifiant possible ? |
|:---|:---|:---|
| `city` | oui : trois aéroports à Paris | ✘ |
| `country` | oui : quatre aéroports en France | ✘ |
| `name` | c'est rare, mais rien ne l'interdit | ✘ (pas garanti) |
| `code` | non : c'est un code de 4 lettres attribué à chaque aéroport du monde par l'organisation internationale de l'aviation civile (son sigle anglais est ICAO), justement pour être unique | ✔ |

Il y a parfois **plusieurs** identifiants possibles : on les appelle les **clés candidates**. Par exemple, deux aéroports ne peuvent pas se trouver exactement au même endroit : leurs coordonnées GPS (latitude et longitude ensemble) les identifient donc, elles aussi. L'identifiant qu'on **choisit** parmi les clés candidates s'appelle la **clé primaire** : ici `code`, le plus simple. C'est pour cela qu'on invente souvent un numéro qui sert de clé : un `id`, le matricule d'un élève, un numéro de commande… Dans un schéma, la clé primaire est **soulignée** : Airports(<u>code</u>, name, type, latitude, altitude, country, city).

**Étape 4 — Faire respecter la clé par le SGBD.** Choisir une clé sur le papier ne suffit pas : il faut la **déclarer** au SGBD, pour qu'il la fasse respecter. La table `PaysEssai` de l'étape 1 (comme `Countries` et `Airports`) a été créée **sans** clé déclarée : c'est pour cela que H2 a accepté la France en double. On crée maintenant une table `Pays`, en déclarant sa clé primaire avec `PRIMARY KEY (colonne)` à la fin du `CREATE TABLE`, et on refait le même essai :

*Dans la console H2 :*

```sql
CREATE TABLE Pays (
    code CHAR(2), name VARCHAR,
    PRIMARY KEY (code) );
INSERT INTO Pays VALUES ('FR', 'France');
INSERT INTO Pays VALUES ('FR', 'France');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Pays ( code CHAR(2), name VARCHAR, PRIMARY KEY (code) );
(Update count: 0)

INSERT INTO Pays VALUES ('FR', 'France');
(Update count: 1)

INSERT INTO Pays VALUES ('FR', 'France');
Erreur : Violation d'index unique ou clé primaire: "PUBLIC.CONSTRAINT_2 INDEX PUBLIC.PRIMARY_KEY_2 ON PUBLIC.PAYS(CODE) VALUES ( /* 1 */ CAST('FR' AS CHAR(2)) )"
Unique index or primary key violation: "PUBLIC.CONSTRAINT_2 INDEX PUBLIC.PRIMARY_KEY_2 ON PUBLIC.PAYS(CODE) VALUES ( /* 1 */ CAST('FR' AS CHAR(2)) )"; SQL statement:
INSERT INTO Pays VALUES ('FR', 'France') [23505-252]
```

- La première insertion est acceptée, la deuxième **refusée** (« violation d'index unique ou clé primaire ») : grâce à la clé primaire, la France ne peut exister qu'une fois dans `Pays`. Cette vérification automatique s'appelle la **contrainte de relation**.
- Une clé peut être faite de **plusieurs** attributs : on les met tous dans la même parenthèse, par exemple `PRIMARY KEY (nom, prenom, date_naissance)`. C'est alors le **groupe** de valeurs qui doit être unique.

---
# 10. Plusieurs tables : la clé étrangère et la jointure

## 10.1 La clé étrangère : une colonne qui renvoie vers une autre table

**Le besoin :** chaque aéroport se trouve dans un pays. On veut que la base le sache, sans recopier dans chaque ligne d'aéroport toutes les informations du pays (son nom, son continent…). On verra dans la partie 11 pourquoi recopier ces informations serait une mauvaise idée.

**L'idée :** dans la table `Airports`, on écrit seulement le **code** du pays, dans la colonne `country`. Ce code suffit pour retrouver **la bonne ligne** dans la table `Countries`, puisque chaque pays a un code différent. Les valeurs `'FR'`, `'ES'`, `'BR'`… de la colonne `country` ne sont donc pas des noms de pays : ce sont des **renvois** vers une ligne de `Countries`. Pour savoir que `'BR'` veut dire *Brazil*, on va voir dans `Countries` la ligne dont le code est `'BR'`.

| Airports (extrait) | | | → | Countries | |
|:---|:---|:---|:---|:---|:---|
| **code** | **name** | **country** | | **code** | **name** |
| LFPG | Charles de Gaulle International Airport | FR | → | FR | France |
| LEMD | Adolfo Suárez Madrid–Barajas Airport | ES | → | ES | Spain |
| SBGR | Guarulhos - Governador André Franco Montoro International Airport | BR | → | BR | Brazil |

Pour que le renvoi désigne **une seule** ligne de `Countries`, il doit viser un identifiant de cette table, c'est-à-dire sa **clé primaire** (partie 9) : ici, le code du pays.

**Définition (polycopié).** Un attribut qui fait référence à la **clé primaire** d'une autre relation est une **clé étrangère**. Ici, `Airports.country` est une clé étrangère qui fait référence à `Countries.code`. Dans un schéma, on la marque d'un astérisque : Airports(<u>code</u>, name, …, country\*, city).

## 10.2 La jointure : recoller deux tables

### Avant de commencer : on travaille sur la base de l'exercice 2

À partir d'ici, **tous les exemples s'exécutent sur la vraie base de l'exercice 2** (les aéroports du monde entier). On ne crée plus aucune table.

- **Si la base de l'exercice 2 est déjà chargée** (question 1 de l'exercice 2), il n'y a rien à faire.
- **Sinon**, on la charge (en remplaçant `/chemin/vers/` par le dossier où se trouve `airports_sql`), puis on exécute les quatre `CREATE INDEX`, **une seule fois** :

```sql
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_tables.sql';
RUNSCRIPT FROM '/chemin/vers/airports_sql/airports_data.sql';
CREATE INDEX ON Countries(code);
CREATE INDEX ON Airports(id);
CREATE INDEX ON Regions(code);
CREATE INDEX ON Runways(id);
```

- Le fichier `airports_tables.sql` commence par `DROP TABLE IF EXISTS Countries;` et `DROP TABLE IF EXISTS Airports;` : les petites tables `Countries` et `Airports` créées dans les parties 4 et 5 sont **remplacées** par les vraies. C'est voulu.

> ⚠️ **Liberté prise :** les `CREATE INDEX` ne sont pas demandés par le sujet, et `CREATE INDEX` est hors programme. Mais sans eux, une jointure sur cette base peut prendre **plusieurs minutes** dans H2. C'est la même consigne que dans la correction de l'exercice 2.

La vraie table `Airports` a **10 colonnes** : `id`, `code`, `name`, `type`, `latitude`, `longitude`, `altitude` (en pieds), `country`, `region`, `city`. La table `Countries` en a **4** : `id`, `code`, `name`, `continent`.

### Le besoin

On veut, pour chaque aéroport, son nom **et le nom de son pays**.

Le problème, c'est que ces deux informations ne sont pas dans la même table :
- le nom de l'aéroport est dans `Airports` ;
- dans `Airports`, le pays n'est écrit que sous la forme d'un **code** (`'FR'`, `'BR'`…), dans la colonne `country` ;
- le **nom** du pays (`France`, `Brazil`…) n'est que dans `Countries`.

Voici un extrait de chacune des deux tables :

*Dans la console H2 :*

```sql
SELECT code, name, country FROM Airports WHERE code = 'LFPG' OR code = 'LEMD' OR code = 'SBGR';
```

*Résultat dans la console H2 :*

```text
CODE | NAME                                                              | COUNTRY
LEMD | Adolfo Suárez Madrid–Barajas Airport                              | ES
LFPG | Charles de Gaulle International Airport                           | FR
SBGR | Guarulhos - Governador André Franco Montoro International Airport | BR
(3 rows)
```

*Dans la console H2 :*

```sql
SELECT * FROM Countries WHERE code = 'FR' OR code = 'ES' OR code = 'BR';
```

*Résultat dans la console H2 :*

```text
ID     | CODE | NAME   | CONTINENT
302791 | BR   | Brazil | SA
302684 | ES   | Spain  | EU
302687 | FR   | France | EU
(3 rows)
```

### Étape 1 : faire le travail à la main, pour un seul aéroport

Pour savoir dans quel pays se trouve l'aéroport de Guarulhos (code `SBGR`), on procède en **deux temps**.

On lit d'abord le code de son pays dans `Airports` :

*Dans la console H2 :*

```sql
SELECT code, name, country FROM Airports WHERE code = 'SBGR';
```

*Résultat dans la console H2 :*

```text
CODE | NAME                                                              | COUNTRY
SBGR | Guarulhos - Governador André Franco Montoro International Airport | BR
(1 row)
```

On va ensuite chercher ce code dans `Countries` :

*Dans la console H2 :*

```sql
SELECT name FROM Countries WHERE code = 'BR';
```

*Résultat dans la console H2 :*

```text
NAME
Brazil
(1 row)
```

Guarulhos est donc au Brésil. On est passé d'une table à l'autre grâce au code `'BR'` : il est écrit dans la colonne `country` de `Airports`, et on le retrouve dans la colonne `code` de `Countries`. C'est le rôle de la **clé étrangère** (partie 10.1).

Cette méthode marche, mais il faudrait la refaire **pour chacun des 73 904 aéroports** de la base. Il faut donc demander au SGBD de faire ce travail lui-même, pour toutes les lignes à la fois. C'est exactement ce que fait une **jointure**.

### Étape 2 : la solution, la jointure, et comment on l'écrit

**La solution :** on demande au SGBD de fabriquer **un seul tableau** qui recolle les deux tables. Chaque ligne d'aéroport y est collée à la ligne de son pays. Pour savoir quelle ligne va avec quelle ligne, on relie la **clé étrangère** de l'une (`Airports.country`) à la **clé primaire** de l'autre (`Countries.code`).

En SQL, cela s'écrit :

```sql
SELECT *
FROM Airports
JOIN Countries ON Airports.country = Countries.code;
```

Chaque morceau dit une chose au SGBD :

| le morceau SQL | ce qu'il dit au SGBD |
|:---|:---|
| `FROM Airports` | de quelle table on part |
| `JOIN Countries` | quelle table on lui colle |
| `ON Airports.country = Countries.code` | **la règle de collage** : quelle ligne de pays va avec quelle ligne d'aéroport |
| `SELECT *` | afficher toutes les colonnes du tableau recollé |

La règle `ON Airports.country = Countries.code` se lit : « colle une ligne d'aéroport avec la ligne de pays dont le `code` est égal au `country` de l'aéroport ». C'est exactement le lien qu'on a utilisé à la main à l'étape 1 : le `BR` de Guarulhos, retrouvé dans la colonne `code` de `Countries`.

On ne lance pas encore cette requête. On regarde d'abord, à l'étape 3, ce que fait le SGBD quand il l'exécute. On la lancera à l'étape 4.

### Étape 3 : ce que fait le SGBD quand il exécute cette requête

Le SGBD fait ce qu'on a fait à la main à l'étape 1, mais pour **chaque ligne** de `Airports` :
1. il lit la valeur de `country` dans cette ligne ;
2. il cherche dans `Countries` la ligne qui a le même `code` ;
3. il **met bout à bout** la ligne de l'aéroport et la ligne du pays trouvée. On obtient **une nouvelle ligne**, qui contient d'abord toutes les colonnes de l'aéroport, puis toutes les colonnes du pays.

**Exemple avec Guarulhos.**

La ligne de Guarulhos dans `Airports` a 10 colonnes :

*Dans la console H2 :*

```sql
SELECT * FROM Airports WHERE code = 'SBGR';
```

*Résultat dans la console H2 :*

```text
ID   | CODE | NAME                                                              | TYPE          | LATITUDE   | LONGITUDE  | ALTITUDE | COUNTRY | REGION | CITY
5910 | SBGR | Guarulhos - Governador André Franco Montoro International Airport | large_airport | -23.431944 | -46.467777 | 2461.0   | BR      | BR-SP  | São Paulo
(1 row)
```

Le SGBD lit `country` = **BR** et cherche dans `Countries` la ligne dont le `code` vaut **BR**. Elle a 4 colonnes :

*Dans la console H2 :*

```sql
SELECT * FROM Countries WHERE code = 'BR';
```

*Résultat dans la console H2 :*

```text
ID     | CODE | NAME   | CONTINENT
302791 | BR   | Brazil | SA
(1 row)
```

Il **recopie ces 4 valeurs à droite des 10 premières**. On obtient une seule ligne de 10 + 4 = **14 colonnes**, qui contient **à la fois** le nom de l'aéroport et le nom de son pays (`Brazil`). On la verra s'afficher à l'étape 4. Les deux tables d'origine, elles, **ne sont pas modifiées** : la jointure fabrique un nouveau tableau, comme tout `SELECT`.

Le SGBD fait la même chose pour les 73 904 aéroports. Par exemple, pour les six grands aéroports situés à plus de 7 000 pieds d'altitude :

| ligne de Airports | son `country` | ligne de Countries trouvée | nouvelle ligne |
|:---|:---|:---|:---|
| Addis Ababa Bole International Airport | ET | ET, Ethiopia | Addis Ababa Bole … ET … Ethiopia |
| Licenciado Benito Juarez International Airport | MX | MX, Mexico | Licenciado Benito Juarez … MX … Mexico |
| Santa Lucía Air Force Base / General Felipe Ángeles International Airport | MX | MX, Mexico | Santa Lucía … MX … Mexico |
| Mariscal Sucre International Airport | EC | EC, Ecuador | Mariscal Sucre … EC … Ecuador |
| El Dorado International Airport | CO | CO, Colombia | El Dorado … CO … Colombia |
| Alejandro Velasco Astete International Airport | PE | PE, Peru | Alejandro Velasco Astete … PE … Peru |

On remarque deux choses :
- la ligne **Mexico** est utilisée **2 fois**, une fois pour chaque aéroport mexicain. Dans le résultat complet, la ligne **France** est utilisée 983 fois, une fois par aéroport français ;
- trois lignes de `Countries` n'ont **aucun** aéroport dans la base : *Pitcairn*, *Tokelau* et *Unknown or unassigned country*. Elles n'apparaissent dans aucune ligne du résultat. Une ligne qui ne trouve pas de partenaire **ne fait pas partie du résultat**.

### Étape 4 : lancer la requête

On lance la requête de l'étape 2. La base contient 73 904 aéroports : pour garder un affichage lisible, on ajoute un `WHERE` (partie 7) qui ne garde que Guarulhos.

*Dans la console H2 :*

```sql
SELECT * FROM Airports JOIN Countries ON Airports.country = Countries.code WHERE Airports.code = 'SBGR';
```

*Résultat dans la console H2 :*

```text
ID   | CODE | NAME                                                              | TYPE          | LATITUDE   | LONGITUDE  | ALTITUDE | COUNTRY | REGION | CITY      | ID     | CODE | NAME   | CONTINENT
5910 | SBGR | Guarulhos - Governador André Franco Montoro International Airport | large_airport | -23.431944 | -46.467777 | 2461.0   | BR      | BR-SP  | São Paulo | 302791 | BR   | Brazil | SA
(1 row)
```

On retrouve la ligne de 14 colonnes de l'étape 3 : les 10 colonnes de `Airports`, puis les 4 colonnes de `Countries`. `COUNTRY` et le deuxième `CODE` ont la même valeur, `BR` : c'est la règle écrite après `ON`. Sans le `WHERE`, H2 afficherait les 73 904 lignes du grand tableau, une par aéroport.

### Étape 5 : ne garder que les colonnes voulues

On veut seulement le nom de l'aéroport et le nom du pays. Mais il y a un piège : le grand tableau contient **deux colonnes `NAME`**, une qui vient de `Airports` et une qui vient de `Countries`. Si on écrit seulement `name`, H2 ne sait pas laquelle on veut :

*Dans la console H2 :*

```sql
SELECT name FROM Airports JOIN Countries ON Airports.country = Countries.code WHERE Airports.code = 'SBGR';
```

*Résultat dans la console H2 :*

```text
Erreur : Nom de colonne ambigu "NAME"
Ambiguous column name "NAME"; SQL statement:
SELECT name FROM Airports JOIN Countries ON Airports.country = Countries.code WHERE Airports.code = 'SBGR' [90059-252]
```

Pour préciser, on écrit **le nom de la table, un point, puis le nom de la colonne**. Cette fois, on garde les six grands aéroports situés à plus de 7 000 pieds d'altitude (environ 2 100 m), ceux du tableau de l'étape 3 :

*Dans la console H2 :*

```sql
SELECT Airports.name, Countries.name
FROM Airports
JOIN Countries ON Airports.country = Countries.code
WHERE Airports.type = 'large_airport' AND Airports.altitude > 7000;
```

*Résultat dans la console H2 :*

```text
NAME                                                                      | NAME
Addis Ababa Bole International Airport                                    | Ethiopia
Licenciado Benito Juarez International Airport                            | Mexico
Santa Lucía Air Force Base / General Felipe Ángeles International Airport | Mexico
Mariscal Sucre International Airport                                      | Ecuador
El Dorado International Airport                                           | Colombia
Alejandro Velasco Astete International Airport                            | Peru
(6 rows)
```

On a la réponse au besoin. H2 affiche deux fois `NAME` en titre de colonne, mais la première colonne vient de `Airports` et la deuxième de `Countries`.

### Étape 6 : pourquoi la règle `ON` est indispensable

Sans la règle, le SGBD ne sait pas quelle ligne coller avec quelle ligne. Il colle alors **chaque aéroport avec chaque pays**. On le voit sur Guarulhos seul :

*Dans la console H2 :*

```sql
SELECT Airports.name, Countries.name FROM Airports JOIN Countries WHERE Airports.code = 'SBGR';
```

*Résultat dans la console H2 (début) :*

```text
NAME                                                              | NAME
Guarulhos - Governador André Franco Montoro International Airport | Andorra
Guarulhos - Governador André Franco Montoro International Airport | United Arab Emirates
Guarulhos - Governador André Franco Montoro International Airport | Afghanistan
Guarulhos - Governador André Franco Montoro International Airport | Antigua and Barbuda
Guarulhos - Governador André Franco Montoro International Airport | Anguilla
Guarulhos - Governador André Franco Montoro International Airport | Albania
Guarulhos - Governador André Franco Montoro International Airport | Armenia
Guarulhos - Governador André Franco Montoro International Airport | Angola
Guarulhos - Governador André Franco Montoro International Airport | Antarctica
Guarulhos - Governador André Franco Montoro International Airport | Argentina
Guarulhos - Governador André Franco Montoro International Airport | American Samoa
Guarulhos - Governador André Franco Montoro International Airport | Austria
...
(248 rows)
```

Guarulhos est collé avec **les 248 pays** de `Countries`, et presque toutes ces lignes sont fausses : Guarulhos n'est pas en Andorre. C'est la règle `ON` qui ne garde que les bonnes associations.

⚠️ **Ne lance pas cette requête sans le `WHERE`** : elle produirait 73 904 aéroports × 248 pays, soit plus de **18 millions** de lignes.

### Étape 7 : une règle fausse donne un résultat faux

La règle doit relier la **clé étrangère** (`Airports.country`) à la **clé primaire** qu'elle désigne (`Countries.code`). Si on se trompe de colonne, par exemple en reliant le code de l'aéroport au code du pays :

*Dans la console H2 :*

```sql
SELECT * FROM Airports JOIN Countries ON Airports.code = Countries.code;
```

*Résultat dans la console H2 :*

```text
ID | CODE | NAME | TYPE | LATITUDE | LONGITUDE | ALTITUDE | COUNTRY | REGION | CITY | ID | CODE | NAME | CONTINENT
(0 rows)
```

On n'obtient **aucune ligne**, parce qu'aucun code d'aéroport (comme `'LFPG'`) n'est égal à un code de pays (comme `'FR'`). H2 n'affiche pas d'erreur : la requête est correcte du point de vue de SQL, mais elle ne répond pas à la question. Il faut donc toujours se demander **quelle colonne renvoie vers quelle table**.

### Étape 8 : après la jointure, tout fonctionne comme avec une seule table

Le grand tableau obtenu se manipule comme une table ordinaire, avec `WHERE`, `ORDER BY`, `COUNT`… Le `WHERE` peut utiliser les colonnes des **deux** tables.

Exemple : *« les grands aéroports d'Océanie, avec leur ville »*. Le continent n'est que dans `Countries`, alors que le type et la ville ne sont que dans `Airports`.

*Dans la console H2 :*

```sql
SELECT Airports.name, Airports.city
FROM Airports
JOIN Countries ON Airports.country = Countries.code
WHERE Countries.continent = 'OC' AND Airports.type = 'large_airport';
```

*Résultat dans la console H2 :*

```text
NAME                                         | CITY
Honiara International Airport                | Honiara
Port Moresby Jacksons International Airport  | Port Moresby
Faa'a International Airport                  | Papeete
Bauerfield International Airport             | Port Vila
Nouméa Magenta Airport                       | Nouméa
Auckland International Airport               | Auckland
Christchurch International Airport           | Christchurch
Wellington International Airport             | Wellington
Antonio B. Won Pat International Airport     | Hagåtña
Brisbane International Airport               | Brisbane
Melbourne International Airport              | Melbourne
Adelaide International Airport               | Adelaide
...
(15 rows)
```

L'ordre est le suivant : le SGBD **colle** d'abord les tables (`FROM … JOIN … ON`), puis il **filtre** les lignes (`WHERE`), et enfin il **choisit** les colonnes (`SELECT`).

### Étape 9 : trois tables, quand il faut faire deux pas

**Le besoin :** on veut, pour chaque piste, sa **longueur** et le **nom du pays** où elle se trouve.

Le problème, c'est que ces deux informations sont dans deux tables qui ne sont pas reliées directement :
- la longueur de la piste est dans `Runways` ;
- le nom du pays est dans `Countries` ;
- `Runways` ne contient **aucune clé étrangère qui fait référence à `Countries`**. Sa seule clé étrangère, `Runways.airport_id`, fait référence à la clé primaire de `Airports` (`Airports.id`) ;
- c'est `Airports` qui contient la clé étrangère vers le pays : `Airports.country`, qui fait référence à la clé primaire de `Countries` (`Countries.code`).

Les clés étrangères forment donc une **chaîne** : `Runways` → `Airports` → `Countries`. Pour aller d'une piste à son pays, il faut faire **deux pas**, donc **deux jointures** :
1. coller chaque piste à son aéroport (`Runways.airport_id = Airports.id`) ;
2. coller chaque aéroport à son pays (`Airports.country = Countries.code`).

Certaines questions de l'exercice 2 demandent exactement cela.

Les pistes sont dans la table `Runways` de l'exercice 2. Ses colonnes sont `id`, `airport_id`, `type`, `length` (longueur en pieds), `width`, `lighted`, `closed`.

**Attention, le renvoi se fait ici par un numéro :** une piste ne contient pas le code de son aéroport (`'LFPG'`), mais son **numéro** `id`, dans la colonne `airport_id`. `Runways.airport_id` est une clé étrangère qui fait référence à `Airports.id`. À la main, pour Charles de Gaulle :

*Dans la console H2 :*

```sql
SELECT id, code, name FROM Airports WHERE code = 'LFPG';
```

*Résultat dans la console H2 :*

```text
ID   | CODE | NAME
4185 | LFPG | Charles de Gaulle International Airport
(1 row)
```

*Dans la console H2 :*

```sql
SELECT * FROM Runways WHERE airport_id = 4185;
```

*Résultat dans la console H2 :*

```text
ID     | AIRPORT_ID | TYPE | LENGTH | WIDTH | LIGHTED | CLOSED
235946 | 4185       | ASP  | 13829  | 148   | 1       | 0
235945 | 4185       | CON  | 8858   | 197   | 1       | 0
235948 | 4185       | ASP  | 8858   | 197   | 1       | 0
235947 | 4185       | ASP  | 13780  | 148   | 1       | 0
(4 rows)
```

Charles de Gaulle a le numéro 4185, et les quatre pistes dont `airport_id` vaut 4185 sont les siennes.

**Premier pas : piste → aéroport.** La règle relie `Runways.airport_id` à `Airports.id`.

*Dans la console H2 :*

```sql
SELECT Runways.length, Airports.name
FROM Runways
JOIN Airports ON Runways.airport_id = Airports.id
WHERE Airports.code = 'LFPG';
```

*Résultat dans la console H2 :*

```text
LENGTH | NAME
13829  | Charles de Gaulle International Airport
8858   | Charles de Gaulle International Airport
8858   | Charles de Gaulle International Airport
13780  | Charles de Gaulle International Airport
(4 rows)
```

Ce sont les quatre pistes trouvées à la main, avec le nom de l'aéroport collé à côté. Les longueurs sont en pieds : 13 829 pieds font environ 4 200 m.

**Deuxième pas : on colle aussi le pays**, avec un deuxième `JOIN` et sa propre règle. On reprend les six grands aéroports situés à plus de 7 000 pieds :

*Dans la console H2 :*

```sql
SELECT Runways.length, Airports.name, Countries.name
FROM Runways
JOIN Airports  ON Runways.airport_id = Airports.id
JOIN Countries ON Airports.country = Countries.code
WHERE Airports.type = 'large_airport' AND Airports.altitude > 7000;
```

*Résultat dans la console H2 :*

```text
LENGTH | NAME                                                                      | NAME
12139  | Addis Ababa Bole International Airport                                    | Ethiopia
12467  | Addis Ababa Bole International Airport                                    | Ethiopia
12966  | Licenciado Benito Juarez International Airport                            | Mexico
12795  | Licenciado Benito Juarez International Airport                            | Mexico
14764  | Santa Lucía Air Force Base / General Felipe Ángeles International Airport | Mexico
14764  | Santa Lucía Air Force Base / General Felipe Ángeles International Airport | Mexico
11483  | Santa Lucía Air Force Base / General Felipe Ángeles International Airport | Mexico
13445  | Mariscal Sucre International Airport                                      | Ecuador
12467  | El Dorado International Airport                                           | Colombia
12467  | El Dorado International Airport                                           | Colombia
11146  | Alejandro Velasco Astete International Airport                            | Peru
(11 rows)
```

Chaque `JOIN` ajoute une table, et chaque table ajoutée a **sa propre règle** `ON`. Les deux règles forment une chaîne : la piste est reliée à l'aéroport (par son numéro), puis l'aéroport est relié au pays (par son code).

### La méthode pour écrire une jointure

Avant d'écrire, on se pose **trois questions** :
1. **Dans quelles tables se trouvent les colonnes dont j'ai besoin ?** Ce sont celles qu'on veut afficher, et aussi celles du `WHERE`.
2. **Comment ces tables sont-elles reliées ?** On cherche la clé étrangère : quelle colonne d'une table contient la clé primaire de l'autre ? Ce lien donne la règle `ON`.
3. **On écrit dans cet ordre** : `FROM` et les `JOIN … ON …` (on colle), puis `WHERE` (on filtre), puis `SELECT` (on choisit les colonnes), en écrivant toujours `table.colonne`.

### Récapitulatif de la syntaxe

Tout ce qu'on vient de voir tient dans ce modèle :
```sql
SELECT table1.colonne, table2.colonne
FROM table1
JOIN table2 ON table1.cle_etrangere = table2.cle_primaire
WHERE condition;
```
- `FROM table1 JOIN table2` : les deux tables à coller (étape 2) ;
- `ON …` : la règle de collage, qui relie la clé étrangère à la clé primaire (étapes 2, 6 et 7) ;
- `WHERE` : le filtre, appliqué après le collage (étape 8) ;
- `SELECT table.colonne` : les colonnes à afficher, avec le nom de la table devant (étape 5).

Avec trois tables, on ajoute un `JOIN … ON …` par table (étape 9) :
```sql
FROM table1
JOIN table2 ON table1.cle_etrangere = table2.cle_primaire
JOIN table3 ON table2.cle_etrangere = table3.cle_primaire
```

✏️ **Vérifie que tu as compris :** pourquoi ne peut-on pas répondre à « les aéroports d'Espagne » avec la seule table `Airports`, en écrivant `WHERE country = 'Spain'` ? *(Réponse : la colonne `country` contient le code `'ES'`, pas le nom `'Spain'` ; le nom n'est que dans `Countries`. Soit on connaît le code et on écrit `country = 'ES'`, soit on fait la jointure.)*

## 10.3 Le SGBD vérifie les clés étrangères : la contrainte de référence

**Le besoin :** une clé étrangère renvoie vers une ligne d'une autre table. Si on écrit dans `Airports` un code de pays qui n'existe pas dans `Countries` (par exemple `'XX'`, à cause d'une faute de frappe), le renvoi ne mène nulle part : l'aéroport est dans un pays « fantôme », et une jointure ne le retrouvera jamais. Il faut donc empêcher qu'une clé étrangère contienne une valeur qui n'existe pas dans la table visée.

**L'idée :** comme pour la clé primaire (partie 9, étape 4), il faut **déclarer** la clé étrangère au SGBD, pour qu'il la fasse respecter. On l'écrit à la fin du `CREATE TABLE` : `FOREIGN KEY (colonne) REFERENCES AutreTable(colonne)`. Le SGBD vérifie alors, à chaque insertion, que la valeur **existe** dans l'autre table.

Les tables de l'exercice 2 ont été créées sans aucune clé déclarée (il n'y a ni `PRIMARY KEY` ni `FOREIGN KEY` dans `airports_tables.sql`) : H2 n'y vérifie rien. Pour voir la vérification, on crée une table `Villes`, dont la colonne `pays` est déclarée comme clé étrangère vers la table `Pays` de la partie 9 (dont la clé primaire est déclarée, et qui ne contient que la France), puis on essaie d'y insérer deux villes :

*Dans la console H2 :*

```sql
CREATE TABLE Villes (
    nom VARCHAR, pays CHAR(2),
    PRIMARY KEY (nom),
    FOREIGN KEY (pays) REFERENCES Pays(code) );
INSERT INTO Villes VALUES ('Lyon', 'FR');
INSERT INTO Villes VALUES ('Rome', 'IT');
```

*Résultat dans la console H2 :*

```text
CREATE TABLE Villes ( nom VARCHAR, pays CHAR(2), PRIMARY KEY (nom), FOREIGN KEY (pays) REFERENCES Pays(code) );
(Update count: 0)

INSERT INTO Villes VALUES ('Lyon', 'FR');
(Update count: 1)

INSERT INTO Villes VALUES ('Rome', 'IT');
Erreur : Intégrité référentielle violation de contrainte: "CONSTRAINT_96: PUBLIC.VILLES FOREIGN KEY(PAYS) REFERENCES PUBLIC.PAYS(CODE) (CAST('IT' AS CHAR(2)))"
Referential integrity constraint violation: "CONSTRAINT_96: PUBLIC.VILLES FOREIGN KEY(PAYS) REFERENCES PUBLIC.PAYS(CODE) (CAST('IT' AS CHAR(2)))"; SQL statement:
INSERT INTO Villes VALUES ('Rome', 'IT') [23506-252]
```

- Lyon est acceptée : le pays `FR` existe dans `Pays`.
- Rome est **refusée** : le pays `IT` n'existe pas dans `Pays`. Sans cette vérification, on aurait une ville dans un pays « fantôme ».

**Conséquence sur l'ordre des opérations :**
- pour **créer et remplir** : d'abord la table **référencée** (`Pays`), ensuite celle qui y fait référence (`Villes`) ;
- pour **supprimer** : l'inverse. On ne peut pas supprimer la France tant que Lyon y fait référence.

**Les trois contraintes d'intégrité** (programme officiel) — ce sont les trois vérifications automatiques du SGBD rencontrées dans ce cours :

| contrainte | ce qui est vérifié | exemple refusé (vu dans ce cours) |
|:---|:---|:---|
| de **domaine** | chaque valeur a le bon type | `' IM '` (4 caractères) dans un `CHAR(2)` — partie 5 |
| de **relation** | pas deux lignes avec la même clé primaire | la France insérée deux fois — partie 9 |
| de **référence** | la valeur d'une clé étrangère existe dans l'autre table | une ville dans le pays `IT` inexistant — partie 10 |

---
# 11. Pourquoi plusieurs tables ? (les anomalies)

**Le besoin :** comprendre pourquoi on s'embête avec deux tables, une clé étrangère et une jointure, au lieu de mettre le nom du pays directement dans la table des aéroports.

**Essayons :** une seule table, avec le nom du pays répété sur chaque ligne.

| code | name | country_name |
|:---|:---|:---|
| LFPG | Charles de Gaulle Airport | France |
| LFPO | Paris-Orly Airport | France |
| LFBO | Toulouse-Blagnac Airport | France |
| LFPI | Paris Issy Heliport | France |

« France » est écrit **4 fois** : c'est une **redondance** (une information répétée). Avec les milliers d'aéroports français d'une vraie base, il serait écrit des milliers de fois. Cela provoque trois sortes de problèmes, les **anomalies** :
- **anomalie de mise à jour** : si on veut corriger l'orthographe d'un pays, il faut modifier **toutes** ses lignes ; si on en oublie une, la base contient deux orthographes différentes pour le même pays ;
- **anomalie d'insertion** : on ne peut pas enregistrer un nouveau pays tant qu'il n'a aucun aéroport (il n'y a pas de ligne où l'écrire) ;
- **anomalie de suppression** : si on supprime le dernier aéroport d'un pays, on perd aussi le nom de ce pays.

**La solution :** découper en deux tables. Le nom de chaque pays est écrit **une seule fois**, dans `Countries` ; les aéroports ne contiennent que le **code** du pays (la clé étrangère). Quand on a besoin des deux informations ensemble, on les **recolle** avec une jointure. C'est le principe des bases de données **relationnelles**.

---
# 12. Modifier et supprimer des lignes : `UPDATE`, `DELETE`

**Le besoin :** les données changent (une information était fausse, un objet n'existe plus), et il faut pouvoir mettre la base à jour. Or les requêtes `SELECT` ne font que **lire**. Pour **changer** le contenu d'une table, il y a trois requêtes : `INSERT` (ajouter une ligne, partie 5), `UPDATE` (modifier des valeurs) et `DELETE` (supprimer des lignes).

### `UPDATE` : modifier des valeurs

**Le besoin :** « L'altitude d'Orly était fausse : c'est 300, pas 291. »

**La syntaxe :**
```sql
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
```

| morceau | sens |
|:---|:---|
| `UPDATE Airports` | on modifie la table `Airports` |
| `SET altitude = 300` | la colonne `altitude` prend la valeur 300 |
| `WHERE code = 'LFPO'` | **seulement** sur les lignes où le code est LFPO |

Le `WHERE` fonctionne exactement comme dans un `SELECT` : il choisit les lignes concernées. On utilise de préférence la **clé primaire**, pour être sûr de ne toucher qu'**une** ligne.

*Dans la console H2 :*

```sql
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
SELECT code, altitude FROM Airports WHERE city = 'Paris' AND type = 'large_airport';
```

*Résultat dans la console H2 :*

```text
UPDATE Airports SET altitude = 300 WHERE code = 'LFPO';
(Update count: 1)

SELECT code, altitude FROM Airports WHERE city = 'Paris' AND type = 'large_airport';
CODE | ALTITUDE
LFPG | 392.0
LFPO | 300.0
(2 rows)
```

`Update count: 1` : une ligne a été modifiée. Le `SELECT` qui suit le confirme.

### `DELETE` : supprimer des lignes

**Le besoin :** « L'héliport d'Issy-les-Moulineaux (code `LFPI`) ne doit plus figurer dans la base. »
```sql
DELETE FROM Airports WHERE code = 'LFPI';
```
Toutes les lignes qui vérifient la condition sont supprimées. Ici, il n'y en a qu'une, puisque chaque aéroport a un code différent.

*Dans la console H2 :*

```sql
DELETE FROM Airports WHERE code = 'LFPI';
SELECT COUNT(*) FROM Airports;
```

*Résultat dans la console H2 :*

```text
DELETE FROM Airports WHERE code = 'LFPI';
(Update count: 1)

SELECT COUNT(*) FROM Airports;
COUNT(*)
73903
(1 row)
```

Il reste 73 903 aéroports au lieu de 73 904. Avec une condition plus large, `DELETE FROM Airports WHERE type = 'heliport';` aurait supprimé d'un coup les **18 705** héliports de la base.

⚠️ **Le piège le plus grave de tout le chapitre : oublier le `WHERE`.**
- `UPDATE Airports SET altitude = 300;` met l'altitude de **tous** les aéroports à 300 ;
- `DELETE FROM Airports;` supprime **tous** les aéroports.

Il n'y a **pas** de bouton « annuler ». Réflexe : avant un `UPDATE` ou un `DELETE`, faire d'abord un `SELECT` avec le même `WHERE`, pour voir quelles lignes vont être touchées.

**Pour remettre la base de l'exercice 2 comme avant** (Orly à 291 pieds, l'héliport d'Issy de retour), on relance les deux `RUNSCRIPT` et les quatre `CREATE INDEX` du début de la partie 10.2 : le fichier `airports_tables.sql` efface les tables, puis `airports_data.sql` les remplit à nouveau.

---
# 13. La méthode pour écrire une requête

**Le besoin :** on connaît maintenant toutes les clauses (`SELECT`, `FROM`, `WHERE`, `JOIN`, `ORDER BY`, `DISTINCT`, les fonctions d'agrégation). Devant une question posée en français, il faut savoir lesquelles utiliser, et dans quel ordre les écrire. On ne commence donc pas à écrire du SQL au hasard : on répond à **quatre questions, dans cet ordre**. Exemple : *« Quels sont les noms des grands aéroports de France, par ordre alphabétique ? »*

| étape | question à se poser | dans l'exemple | ce qu'on écrit |
|:---|:---|:---|:---|
| 1 | Qu'est-ce qu'on veut **afficher** ? | les noms | `SELECT name` |
| 2 | Dans quelle(s) **table(s)** sont les informations (à afficher **et** à tester) ? | tout est dans `Airports` | `FROM Airports` (avec `JOIN … ON …` s'il faut deux tables) |
| 3 | Quelles **lignes** garde-t-on ? | celles de France, de type grand aéroport | `WHERE country = 'FR' AND type = 'large_airport'` |
| 4 | Faut-il **compter, trier, enlever les doublons** ? | trier par nom | `ORDER BY name` |

Résultat : `SELECT name FROM Airports WHERE country = 'FR' AND type = 'large_airport' ORDER BY name;`

**Traduire les mots de la question :**

| dans la question | en SQL |
|:---|:---|
| « quels sont les… », « donner les… » | `SELECT` les colonnes demandées |
| « de France », « dont l'altitude dépasse… » | `WHERE` |
| « et », « ou » | `AND`, `OR` |
| « contient le mot… », « commence par… » | `LIKE '%mot%'`, `LIKE 'mot%'` |
| « combien » | `COUNT(*)` |
| « la moyenne », « le total » | `AVG`, `SUM` |
| « le plus grand », « le plus petit » | `MAX`, `MIN` (ou `ORDER BY`) ; « le **nom** du plus… » → sous-requête |
| « par ordre alphabétique », « classés par… » | `ORDER BY` (`DESC` pour décroissant) |
| « les différents… », « sans répétition » | `DISTINCT` |
| l'information est dans deux tables | `JOIN … ON clé_étrangère = clé_primaire` |

**Conseils pratiques :**
- commencer par `SELECT * FROM table;` pour voir comment les valeurs sont écrites (`'heliport'` ? `'FR'` ou `'France'` ?) ;
- construire la requête **petit à petit** : d'abord sans `WHERE`, puis ajouter le `WHERE`, puis le tri ; vérifier le résultat à chaque étape.

## Fiche récapitulative

Dans cette fiche, `T` et `U` désignent des noms de tables, et `a`, `b`, `c`, `d` des noms de colonnes.

```sql
-- créer, remplir
CREATE TABLE T (a INT, b VARCHAR, PRIMARY KEY (a), FOREIGN KEY (b) REFERENCES U(c));
INSERT INTO T (a, b) VALUES (1, 'texte');
RUNSCRIPT FROM 'chemin/vers/fichier.sql';

-- interroger (ordre d'écriture imposé)
SELECT a, b FROM T WHERE a > 3 AND b LIKE '%mot%' ORDER BY a DESC;
SELECT DISTINCT b FROM T;
SELECT COUNT(*), SUM(a), AVG(a), MIN(a), MAX(a) FROM T WHERE ...;
SELECT b FROM T WHERE a = (SELECT MAX(a) FROM T);

-- plusieurs tables
SELECT T.a, U.d FROM T JOIN U ON T.b = U.c WHERE ...;

-- modifier, supprimer (toujours avec WHERE !)
UPDATE T SET b = 'nouveau' WHERE a = 1;
DELETE FROM T WHERE a = 1;
```

| mot | sens |
|:---|:---|
| relation / attribut / enregistrement | table / colonne / ligne |
| domaine | type des valeurs d'un attribut |
| schéma | liste des attributs et de leurs types, clé primaire soulignée, clés étrangères marquées |
| clé primaire | identifie chaque ligne de façon unique |
| clé étrangère | fait référence à la clé primaire d'une autre table |
| jointure | recolle deux tables grâce à une clé étrangère |
| SGBD | logiciel qui gère la base : persistance, accès concurrents, efficacité, sécurité |

---
# 14. *(Pour le professeur)* Quelle partie pour quel exercice du polycopié

| exercice du polycopié | parties du cours à avoir lues |
|:---|:---|
| Ex. 1 : création d'une table, import des données | 1 à 6 |
| Ex. 2 : aéroports | 7, 8 et 10 (jointures à partir de la question 3) |
| Ex. 3 : Vélib' | 7, 8 et 10 |
| Ex. 4 et 5 : clés primaires | 9 |
| Ex. 6 : achats, clés étrangères | 9 et 10 |
| Ex. 7 : location de voitures | toutes, dont 12 |

> ⚠️ **Liberté prise :** dans les parties 1 à 9, les données des exemples (quelques pays et aéroports) sont une petite sélection faite pour ce cours. À partir de la partie 10.2, les exemples utilisent la vraie base de l'exercice 2. Les résultats ont été obtenus avec H2 et recopiés, sans la durée d'exécution, avec « Erreur : » à la place du nom technique de l'erreur Java, et, pour les résultats longs, avec seulement les 12 premières lignes, suivies de « ... » et du nombre total de lignes.